In [ ]:
!pip install pandas python-dotenv
!python -m pip install google-genai==1.53.0
import json
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple
import pandas as pd
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv, find_dotenv
import time
load_dotenv(find_dotenv()) 

In [ ]:
from pydantic import BaseModel
from typing import List, Literal


class ValidationSchema(BaseModel):
    validation_present: bool

    evidence_sources: List[
        Literal["pipeline", "description", "comments", "code_diff"]
    ]

    validation_type: Literal[
        "benchmark",
        "profiling",
        "static-analysis",
        "anecdotal",
        "none"
    ]

    validation_description: str
    pipeline_signal: str
    description_signal: str
    comment_signal: str

In [ ]:
counter = 0
def run_llm(prompt: str, user_prompt: str) -> ValidationSchema:
    model = "gemini-3-pro-preview"
    global counter
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
    GEMINI_API_KEY_2 = os.getenv("GEMINI_API_KEY_2")
    client = genai.Client(api_key=GEMINI_API_KEY)
    client_2 = genai.Client(api_key=GEMINI_API_KEY_2)
    counter += 1

    config = types.GenerateContentConfig(
            temperature=0.0,
            response_mime_type="application/json",
            response_json_schema= ValidationSchema.model_json_schema(),
            thinking_config=types.ThinkingConfig(
                thinking_level=types.ThinkingLevel.HIGH
            ),
            system_instruction=prompt
    )
    
    try:
        response = None    
        if counter >= 230:   
            response = client_2.models.generate_content(
                model=model,
                contents=user_prompt,
                config=config,
        )
        else:
            response = client.models.generate_content(
                model=model,
                contents=user_prompt,
                config=config,
        )  
     
        return ValidationSchema.model_validate_json(response.text)
    except Exception as e:
        raise e

In [ ]:
def find_datasets_dir(start: Optional[Path] = None) -> Path:
    start = start or Path.cwd()
    for path in (start, *start.parents):
        candidate = path / "datasets"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find 'datasets' directory from {start}")


DATASETS_DIR = find_datasets_dir()
PROJECT_ROOT = DATASETS_DIR.parent

def truncate(text: str, limit: int = 15000) -> str:
    return text if len(text) <= limit else text[:limit] + "...[truncated]"

def load_pr_core(prefix: str) -> pd.DataFrame:
    core = pd.read_parquet(
        DATASETS_DIR / f"{prefix}_pr" / f"{prefix}_pr_commit_details.parquet"
    )
    return core.set_index("pr_id")


def collect_comments(prefix: str, pr_id: int) -> List[str]:
    issue = pd.read_parquet(
        DATASETS_DIR / f"{prefix}_pr" / f"{prefix}_pr_issue_comments.parquet"
    )
    review = pd.read_parquet(
        DATASETS_DIR / f"{prefix}_pr" / f"{prefix}_pr_review_comments.parquet"
    )
    texts = []
    for df in (issue, review):
        subset = df[df["pr_id"] == pr_id]
        texts.extend(subset["body"].dropna().tolist())
    return texts


def collect_pipeline_names(prefix: str, pr_id: int) -> List[str]:
    workflows = pd.read_parquet(
        DATASETS_DIR / f"{prefix}_pr" / f"{prefix}_pr_workflow_runs.parquet"
    )
    subset = workflows[workflows["pr_id"] == pr_id]
    return sorted(subset["workflow_name"].dropna().unique().tolist())

def pr_ids_from_commits(prefix: str, limit: Optional[int] = None) -> Iterable[int]:
    commits = pd.read_parquet(
        DATASETS_DIR / f"{prefix}_pr" / f"{prefix}_pr_commits.parquet"
    )
    pr_ids = sorted(commits["pr_id"].dropna().astype(int).unique().tolist())
    return pr_ids if limit is None else pr_ids[:limit]


In [ ]:
from typing import Dict, Tuple
import pandas as pd

def analyze_pr(
    prefix: str,
    pr_id: int,
    author_type: str,
    pr_core: pd.DataFrame,
) -> Tuple[Dict, Dict]:
    # pr_core ahora es *_commit_details, puede haber varias filas por pr_id
    row = pr_core.loc[pr_id]

    # Si solo hay una fila, loc devuelve Series → lo convertimos a DataFrame
    if isinstance(row, pd.Series):
        row = row.to_frame().T

    # ====== PIPELINES & COMMENTS ======
    pipeline_names = collect_pipeline_names(prefix, pr_id)
    comments = collect_comments(prefix, pr_id)

    # ====== DESCRIPTION (una por PR) ======
    description = ""
    if "pr_description" in row.columns:
        desc_series = row["pr_description"].dropna().astype(str)
        if not desc_series.empty:
            description = desc_series.iloc[0].strip()

    # ====== CODE DIFF (patches + fallback a filenames) ======
    code_diff = ""
    if "patch" in row.columns:
        patches = row["patch"].dropna().astype(str)
        if not patches.empty:
            # concatenamos todos los patches del PR
            code_diff = "\n\n".join(patches.tolist())
        elif "filename" in row.columns:
            filenames = row["filename"].dropna().astype(str)
            if not filenames.empty:
                code_diff = "[NO PATCH AVAILABLE] Files touched:\n" + "\n".join(
                    filenames.tolist()
                )
    # Si no hay columna patch ni filename, code_diff se queda en ""

    DEVELOPER_PROMPT_TEMPLATE = """
    You are a classifier for performance validation evidence in GitHub Pull Requests (PRs).

    Your job is to decide whether there is explicit performance validation, and if so,
    to classify it into EXACTLY ONE validation_type category based on the definitions below.

    First, carefully read and internalize these validation_type categories:

    1. Benchmark-Based Validation (Unit Tests or Microbenchmarks)
    Definition:
    The PR validates performance by running benchmark tests—either existing unit tests or newly added benchmark/microbenchmark tests. It includes explicit before-and-after comparisons such as runtime, throughput, memory, CPU usage, or any quantitative metric collected from tests.

    2. Profiling-Based Validation (Application- or Function-Level Profiling)
    Definition:
    The PR uses profiling tools to validate performance, typically capturing stack samples, CPU hotspots, flamegraphs, or function-level timings. Evidence includes profiling outputs before and after the change.

    3. Static-Analysis-Based Validation (Reasoning Without Runtime Evidence)
    Definition:
    The PR argues for performance improvement through static reasoning about the code—algorithmic complexity, data-structure changes, loop bounds, allocation count reduction, etc., without providing runtime/profiling data.

    4. Anecdotal or Informal Local Testing (No Evidence Provided)
    Definition:
    The PR claims that performance is improved based on local testing, intuition, or manual observation, but provides no quantitative metrics, no profiling output, and no static-analysis justification.

    Decision Rules:

    1) First decide if performance validation is explicitly present (validation_present).
    - Set validation_present = TRUE only when the PR explicitly shows some form of validation evidence: benchmarks/microbenchmarks, profiling traces, static performance reasoning, or explicit local testing statements.
    - Do NOT infer validation from performance intent alone.

    2) If validation_present is "FALSE", please set validation_type = "none", also set evidence_sources = [] also set validation_description to explain the absence of validation. Never assign any other validation_type.
    
    3) If validation_present is "TRUE", please choose exactly ONE non-"none" validation_type from the following: Benchmark-Based Validation; Profiling-Based Validation; Static-Analysis-Based Validation; Anecdotal or Informal Local Testing

    4) evidence_sources must list where the validation is explicitly mentioned: "pipeline", "description", "comments", "code_diff"

    5) When validation_type is "benchmark", "profiling": mention the metrics used (latency, throughput, memory, CPU, etc.).

    You must ALWAYS return STRICT JSON with exactly these keys: validation_present, evidence_sources, validation_type, metrics, validation_description, pipeline_signal, description_signal, comment_signal.

    No extra commentary. No markdown. No explanations.
    """

    USER_PROMPT_TEMPLATE = """
    Classify the following PR strictly using the rules and definitions from the system.

    PIPELINES:
    {pipeline_names}

    DESCRIPTION:
    {description}

    COMMENTS:
    {comments}
    
    CODE DIFF:
    {code_diff}
    """

    # Ahora description y code_diff son siempre strings, no Series
    if not pipeline_names and not description and not comments and not code_diff:
        print(f"Short-circuiting PR {pr_id} with no signals")
        print(f"pipeline_names: {pipeline_names}")
        print(f"description: {description}")
        print(f"comments: {comments}")
        print(f"code_diff: {code_diff}")
        raise Exception("No signals present")

    prompt = USER_PROMPT_TEMPLATE.format(
        pipeline_names="- " + "- ".join(pipeline_names) if pipeline_names else "None",
        description=truncate(description) if description else "None",
        comments="- " + "- ".join(truncate(" | ".join(comments)).split(" | ")) if comments else "None",
        code_diff=truncate(code_diff) if code_diff else "None",
    )

    developer_prompt = DEVELOPER_PROMPT_TEMPLATE

    raw_openai = ""
    try:
        raw_openai = run_llm(
            prompt=developer_prompt,
            user_prompt=prompt,
        ).model_dump()
    except Exception as exc:
        raise exc

    parsed_openai = raw_openai

    evidence_sources = parsed_openai.get("evidence_sources") or []
    if isinstance(evidence_sources, (tuple, list)):
        evidence_sources = list(evidence_sources)
    else:
        evidence_sources = []

    # Tomamos metadatos del PR de la primera fila
    repo_owner = row["repo_owner"].iloc[0] if "repo_owner" in row.columns else ""
    repo_name = row["repo_name"].iloc[0] if "repo_name" in row.columns else ""
    pr_number = row["pr_number"].iloc[0] if "pr_number" in row.columns else None
    pr_title = row["pr_title"].iloc[0] if "pr_title" in row.columns else ""

    result_openai = {
        "pr_id": pr_id,
        "author_type": author_type,
        "repo": f"{repo_owner}/{repo_name}",
        "pr_number": pr_number,
        "pr_title": pr_title,
        "pipeline_names": pipeline_names,
        "validation_present": parsed_openai.get("validation_present", False),
        "evidence_sources": evidence_sources,
        "validation_type": parsed_openai.get("validation_type", "none"),
        "validation_description": parsed_openai.get("validation_description", "No validation evidence"),
        "pipeline_signal": parsed_openai.get("pipeline_signal", ""),
        "description_signal": parsed_openai.get("description_signal", ""),
        "comment_signal": parsed_openai.get("comment_signal", ""),
    }

    return result_openai

In [ ]:
out_dir = PROJECT_ROOT / "RQ3"
out_dir.mkdir(exist_ok=True, parents=True)

out_path = out_dir / "rq3_validation_evidence_gemini.parquet"
error_csv_path = out_dir / "rq3_validation_errors_gemini.csv"

ai_core = load_pr_core("ai")
human_core = load_pr_core("human")

limit = None

if out_path.exists():
    df_existing = pd.read_parquet(out_path)
    records = df_existing.to_dict("records")
    processed_ids = set(df_existing["pr_id"].astype(int).tolist())
    print(f"[resume] Loaded {len(records)} existing records")
else:
    records = []
    processed_ids = set()
    print("[resume] No previous file found, starting fresh")


def save_partial(records, out_path):
    if not records:
        return
    df_tmp = pd.DataFrame(records)
    df_tmp.to_parquet(out_path, index=False)
    # print(f"[partial save] Saved {len(df_tmp)} rows to {out_path}")


def log_error(pr_id, prefix, author_type, exc):
    row = {
        "prefix": prefix,
        "pr_id": pr_id,
        "author_type": author_type,
        "error": str(exc),
    }
    df_err = pd.DataFrame([row])
    header = not error_csv_path.exists()
    df_err.to_csv(error_csv_path, mode="a", header=header, index=False)
    print(f"[error] Logged PR {pr_id} ({prefix}/{author_type}) to {error_csv_path}: {exc}")


ai_ids = list(pr_ids_from_commits("ai", limit=limit))
human_ids = list(pr_ids_from_commits("human", limit=limit))
print(f"Processing {len(ai_ids)} AI PRs and {len(human_ids)} human PRs (first {limit} each).")



for idx, pr_id in enumerate(ai_ids, 1):

    if pr_id in processed_ids:
        print(f"[skip] AI PR {idx}/{len(ai_ids)} already processed: {pr_id}")
        continue

    print(f"Processing AI PR {idx}/{len(ai_ids)}: {pr_id}")

    try:
        res = analyze_pr("ai", pr_id, "ai_agent", ai_core)
        records.append(res)

    except Exception as exc:
        log_error(pr_id, prefix="ai", author_type="ai_agent", exc=exc)

    # ---- SAVE EVERY 1 ----
    if len(records) % 1 == 0:
        save_partial(records, out_path)



for idx, pr_id in enumerate(human_ids, 1):

    if pr_id in processed_ids:
        print(f"[skip] human PR {idx}/{len(human_ids)} already processed: {pr_id}")
        continue

    print(f"Processing human PR {idx}/{len(human_ids)}: {pr_id}")

    try:
        res = analyze_pr("human", pr_id, "human", human_core)
        records.append(res)

    except Exception as exc:
        log_error(pr_id, prefix="human", author_type="human", exc=exc)

    # ---- SAVE EVERY 1 ----
    if len(records) % 1 == 0:
        save_partial(records, out_path)


# ============================
# Final save ✅
# ============================
df = pd.DataFrame(records)
df.to_parquet(out_path, index=False)
print(f"Saved FINAL GEMINI {len(df)} rows to {out_path}")
print(f"Errored PRs (if any) logged to {error_csv_path}")